# SL-7 Cross-Model Run — Phi-3.5-Vision-Instruct (family-gated protocol)

Model: `microsoft/Phi-3.5-vision-instruct`

Runs the **unchanged frozen SL-7 v16 benchmark**: 20 calibration items + 150 scored items.

**Cross-model protocol:** fresh context per item, greedy decoding, max 64 new tokens, batch size 1, unchanged images/prompts/keys/family thresholds. Calibration is applied **per prerequisite skill**: a failed calibration skill makes only dependent blind-spot families ineligible for headline confirmation; the remaining families still run.

**Phi-native visual preprocessing:** following Microsoft's model guidance, this notebook uses `num_crops=16` for single-image items and `num_crops=4` for multi-image items. This rule is fixed before running calibration/scored items and is not changed based on results.


In [1]:
%pip install -q -U "transformers==5.17.0" "accelerate>=1.0" "huggingface_hub>=0.30" "pillow"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 25.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
PROTOCOL_VERSION="SL7_CROSS_MODEL_FAMILY_GATED_V2"
print("="*78)
print(PROTOCOL_VERSION)
print("Model: microsoft/Phi-3.5-vision-instruct")
print("Single image: num_crops=16 | Multi-image: num_crops=4")
print("Calibration failures gate only dependent families; the 150-item holdout continues.")
print("="*78)


SL7_CROSS_MODEL_FAMILY_GATED_V2
Model: microsoft/Phi-3.5-vision-instruct
Single image: num_crops=16 | Multi-image: num_crops=4
Calibration failures gate only dependent families; the 150-item holdout continues.


In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
import glob, csv, json, time, platform, sys, zipfile, importlib, random, gc
from pathlib import Path
from PIL import Image
import torch, transformers

MAX_NEW_TOKENS=64
INFERENCE_SEED=20261004
random.seed(INFERENCE_SEED); torch.manual_seed(INFERENCE_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(INFERENCE_SEED)

hits=[]
for p in glob.glob("/kaggle/input/**/dataset/items.csv", recursive=True):
    root=Path(p).parent.parent
    if (root/"VERSION_SL7_v16.txt").exists() and (root/"eval/score.py").exists():
        hits.append(root)
if not hits:
    raise FileNotFoundError("Add the snakes-ladders-SL7-multimodel-v1 package as a Kaggle input dataset.")
ROOT=hits[0]
ITEMS=list(csv.DictReader(open(ROOT/"dataset/items.csv",encoding="utf-8")))
CAL=[x for x in ITEMS if x["part"]=="CAL"]
SCORED=[x for x in ITEMS if x["part"]=="SCORED"]
print("Frozen project root:",ROOT)
print("Calibration:",len(CAL),"Scored:",len(SCORED))

sys.path.insert(0,str(ROOT/"eval"))
import score
importlib.reload(score)


Frozen project root: /kaggle/input/datasets/sabahnushra/snake-ladder-multimodel-dataset/snakes-ladders-SL7-multimodel-v1
Calibration: 20 Scored: 150


<module 'score' from '/kaggle/input/datasets/sabahnushra/snake-ladder-multimodel-dataset/snakes-ladders-SL7-multimodel-v1/eval/score.py'>

In [4]:
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID='microsoft/Phi-3.5-vision-instruct'
MODEL_SLUG='phi3_5_vision'
MODEL_DISPLAY='Phi-3.5-Vision-Instruct'
OUT=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}")
OUT.mkdir(exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle GPU accelerator.")

# Eager attention is the safe T4 path when flash-attn is unavailable.
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"":0},
    trust_remote_code=True,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    _attn_implementation="eager",
)
model.eval()

# Fixed, model-native preprocessing rule declared before evaluation.
processor_single=AutoProcessor.from_pretrained(MODEL_ID,trust_remote_code=True,num_crops=16)
processor_multi=AutoProcessor.from_pretrained(MODEL_ID,trust_remote_code=True,num_crops=4)

free,total=torch.cuda.mem_get_info()
print(f"GPU after model load: {(total-free)/2**30:.2f} GiB used / {total/2**30:.2f} GiB total")

env={
 "model_key":MODEL_SLUG,"model_display":MODEL_DISPLAY,"model_id":MODEL_ID,
 "parameters":sum(p.numel() for p in model.parameters()),"transformers":transformers.__version__,
 "torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0),
 "python":platform.python_version(),"dtype":"float16","attention":"eager",
 "decoding":"greedy","max_new_tokens":MAX_NEW_TOKENS,"inference_seed":INFERENCE_SEED,
 "dataset":"SL7_v16_frozen","phi_single_image_num_crops":16,"phi_multi_image_num_crops":4,
 "generation_use_cache":False,
}
(OUT/"environment.json").write_text(json.dumps(env,indent=2))
print(json.dumps(env,indent=2))


config.json:   0%|          | 0.00/3.78k [00:00<?, ?B/s]

configuration_phi3_v.py:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-vision-instruct:
- configuration_phi3_v.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'


modeling_phi3_v.py:   0%|          | 0.00/88.9k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-vision-instruct:
- modeling_phi3_v.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'


GPU after model load: 8.01 GiB used / 14.56 GiB total
{
  "model_key": "phi3_5_vision",
  "model_display": "Phi-3.5-Vision-Instruct",
  "model_id": "microsoft/Phi-3.5-vision-instruct",
  "parameters": 4146621440,
  "transformers": "5.17.0",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "python": "3.13.15",
  "dtype": "float16",
  "attention": "eager",
  "decoding": "greedy",
  "max_new_tokens": 64,
  "inference_seed": 20261004,
  "dataset": "SL7_v16_frozen",
  "phi_single_image_num_crops": 16,
  "phi_multi_image_num_crops": 4,
  "generation_use_cache": false
}


In [5]:
def run_item(it):
    rels=[p for p in it["image_paths"].split("|") if p]
    images=[]
    for rel in rels:
        with Image.open(ROOT/rel) as im:
            images.append(im.convert("RGB").copy())

    # Microsoft Phi-3.5-Vision uses explicit numbered image placeholders.
    placeholder="".join(f"<|image_{i}|>\n" for i in range(1,len(images)+1))
    processor = processor_single if len(images)<=1 else processor_multi
    messages=[{"role":"user","content":placeholder+it["prompt"]}]
    prompt=processor.tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    t0=time.time()
    inputs=processor(prompt,images,return_tensors="pt").to("cuda:0")
    try:
        with torch.inference_mode():
            generated=model.generate(
                **inputs,
                eos_token_id=processor.tokenizer.eos_token_id,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=False,
            )
        new=generated[:,inputs["input_ids"].shape[1]:]
        reply=processor.batch_decode(new,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip()
        return {"item_id":it["item_id"],"raw_output":reply,"seconds":round(time.time()-t0,2)}
    finally:
        gc.collect()
        torch.cuda.empty_cache()


In [6]:
RAW=OUT/"raw_outputs.csv"
outrows=[]

def save_raw():
    with open(RAW,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["item_id","raw_output","seconds"])
        w.writeheader(); w.writerows(outrows)

for idx,it in enumerate(CAL,1):
    outrows.append(run_item(it)); save_raw()
    print(f"CAL {idx}/{len(CAL)}",it["item_id"],"=>",outrows[-1]["raw_output"][:160])

pred={r["item_id"]:r["raw_output"] for r in outrows}
groups={}
for it in CAL:
    parsed=score.parse_strict(pred[it["item_id"]],it["answer_format"])
    ok=parsed.lower()==it["correct_answer"].lower() if parsed else False
    groups.setdefault(it["task_family"],[]).append(ok)
cal_counts={k:(sum(v),len(v)) for k,v in groups.items()}
requirements={"CAL_number":7,"CAL_endpoint":3,"CAL_die":4,"CAL_multi_image":3}
cal_pass={k:(sum(groups.get(k,[]))>=need) for k,need in requirements.items()}
failed=[k for k,v in cal_pass.items() if not v]

family_gate_requirements={
 "B1_zigzag_direction":["CAL_number"],
 "B2_hidden_reconstruction":["CAL_number"],
 "B3_endpoint_anchoring":["CAL_endpoint"],
 "B4_perception_to_action":["CAL_number","CAL_endpoint"],
 "B5_multi_image_position":["CAL_number","CAL_endpoint","CAL_multi_image"],
 "B6_conditioned_counting":["CAL_number","CAL_endpoint"],
 "B7_multi_source_composition":["CAL_number","CAL_die","CAL_multi_image"],
}
family_eligibility={fam:all(cal_pass[g] for g in gates) for fam,gates in family_gate_requirements.items()}

cal_record={
 "counts":{k:list(v) for k,v in cal_counts.items()},"requirements":requirements,
 "calibration_pass":cal_pass,"original_global_gate_passed":not failed,
 "failed_calibrations":failed,"cross_model_protocol":"family_specific_calibration_gating_v2",
 "family_gate_requirements":family_gate_requirements,"family_eligibility":family_eligibility,
}
(OUT/"calibration_status.json").write_text(json.dumps(cal_record,indent=2))

print("\nCalibration:",cal_counts)
print("Calibration pass map:",cal_pass)
print("\nFamily eligibility:")
for fam,ok in family_eligibility.items():
    print(" ",fam,":","ELIGIBLE" if ok else "INELIGIBLE / diagnostic only",family_gate_requirements[fam])
if failed:
    print("\nOriginal global gate would fail for:",failed)
    print("FAMILY-GATED MODE: continuing to the frozen 150-item holdout.")
else:
    print("\nAll calibration gates passed; all seven families are eligible.")


/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

CAL 1/20 CAL_NUM_01 => 7
CAL 2/20 CAL_NUM_02 => 15
CAL 3/20 CAL_NUM_03 => 24
CAL 4/20 CAL_NUM_04 => 33
CAL 5/20 CAL_NUM_05 => 45
CAL 6/20 CAL_NUM_06 => 58
CAL 7/20 CAL_NUM_07 => 74
CAL 8/20 CAL_NUM_08 => 97
CAL 9/20 CAL_END_01 => snake tail
CAL 10/20 CAL_END_02 => snake head
CAL 11/20 CAL_END_03 => snake tail
CAL 12/20 CAL_END_04 => snake tail
CAL 13/20 CAL_DIE_01 => 1
CAL 14/20 CAL_DIE_02 => 4
CAL 15/20 CAL_DIE_03 => 5
CAL 16/20 CAL_DIE_04 => 3
CAL 17/20 CAL_CMP_01 => B
CAL 18/20 CAL_CMP_02 => B
CAL 19/20 CAL_CMP_03 => B
CAL 20/20 CAL_CMP_04 => B

Calibration: {'CAL_number': (8, 8), 'CAL_endpoint': (0, 4), 'CAL_die': (3, 4), 'CAL_multi_image': (2, 4)}
Calibration pass map: {'CAL_number': True, 'CAL_endpoint': False, 'CAL_die': False, 'CAL_multi_image': False}

Family eligibility:
  B1_zigzag_direction : ELIGIBLE ['CAL_number']
  B2_hidden_reconstruction : ELIGIBLE ['CAL_number']
  B3_endpoint_anchoring : INELIGIBLE / diagnostic only ['CAL_endpoint']
  B4_perception_to_action : INELIGI

In [7]:
for idx,it in enumerate(SCORED,1):
    outrows.append(run_item(it)); save_raw()
    if idx%10==0 or idx==len(SCORED): print(f"SCORED {idx}/{len(SCORED)} complete")
print("Raw outputs saved:",RAW)


SCORED 10/150 complete
SCORED 20/150 complete
SCORED 30/150 complete
SCORED 40/150 complete
SCORED 50/150 complete
SCORED 60/150 complete
SCORED 70/150 complete
SCORED 80/150 complete
SCORED 90/150 complete
SCORED 100/150 complete
SCORED 110/150 complete
SCORED 120/150 complete
SCORED 130/150 complete
SCORED 140/150 complete
SCORED 150/150 complete
Raw outputs saved: /kaggle/working/results_SL7_phi3_5_vision/raw_outputs.csv


In [8]:
# Scorer presentation patch for n=0 families; no item score or threshold changes.
_original_wilson=score.wilson
def _wilson_zero_safe(k,n,*args,**kwargs):
    if n==0: return (float("nan"),float("nan"))
    return _original_wilson(k,n,*args,**kwargs)
score.wilson=_wilson_zero_safe

scored_rows,family_rows,original_cal_pass=score.score(ROOT/"dataset/items.csv",RAW,OUT)
_summary=OUT/"summary.md"
if _summary.exists():
    _txt=_summary.read_text(encoding="utf-8").replace("nan–nan%","N/A").replace("nan-nan%","N/A")
    _summary.write_text(_txt,encoding="utf-8")

fam_csv=OUT/"family_summary.csv"
fam_rows=list(csv.DictReader(open(fam_csv,encoding="utf-8")))
for r in fam_rows:
    fam=r["family"]
    eligible=bool(family_eligibility.get(fam,False))
    frozen=(r["confirmed"]=="1")
    r["calibration_eligible"]="1" if eligible else "0"
    r["headline_confirmed"]="1" if (eligible and frozen) else "0"
    reqs=family_gate_requirements.get(fam,[])
    failed_reqs=[g for g in reqs if not cal_pass.get(g,False)]
    r["eligibility_note"]="eligible" if eligible else "failed calibration: "+",".join(failed_reqs)

fg_csv=OUT/"family_summary_family_gated.csv"
with open(fg_csv,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=list(fam_rows[0].keys())); w.writeheader(); w.writerows(fam_rows)

L=["# SL-7 Cross-Model Family-Gated Summary","",f"Model: **{MODEL_DISPLAY}**","",
   "Frozen SL-7 v16 stimuli and family thresholds are unchanged. Calibration eligibility is applied per dependent family.","",
   "## Calibration","", "| Calibration | Correct | Required | Pass |","|---|---:|---:|---|"]
for name in ["CAL_number","CAL_endpoint","CAL_die","CAL_multi_image"]:
    k,n=cal_counts.get(name,(0,0)); need=requirements[name]
    L.append(f"| {name.replace('CAL_','')} | {k}/{n} | {need}/{n} | {'PASS' if cal_pass[name] else 'FAIL'} |")
L += ["","## Family results","",
      "| Family | Eligible? | Frozen confirmed? | Headline confirmed? | Control/component | Probe | Gap | Note |",
      "|---|---|---|---|---:|---:|---:|---|"]
for r in fam_rows:
    elig="YES" if r["calibration_eligible"]=="1" else "NO"
    conf="YES" if r["confirmed"]=="1" else "NO"
    head="YES" if r["headline_confirmed"]=="1" else "NO"
    L.append(f"| {r['family']} | {elig} | {conf} | **{head}** | {r['control_or_component_correct']}/{r['control_or_component_n']} | {r['probe_correct']}/{r['probe_n']} | {r['gap_percentage_points']} pp | {r['eligibility_note']} |")
L += ["","`headline_confirmed=1` requires both the frozen family criterion and all directly relevant calibration prerequisites.",""]
summary_fg=OUT/"summary_family_gated.md"
summary_fg.write_text("\n".join(L),encoding="utf-8")
print(summary_fg.read_text())

zip_path=Path(f"/kaggle/working/results_SL7_{MODEL_SLUG}_family_gated.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.name)
print("Download:",zip_path)


# SL-7 Cross-Model Family-Gated Summary

Model: **Phi-3.5-Vision-Instruct**

Frozen SL-7 v16 stimuli and family thresholds are unchanged. Calibration eligibility is applied per dependent family.

## Calibration

| Calibration | Correct | Required | Pass |
|---|---:|---:|---|
| number | 8/8 | 7/8 | PASS |
| endpoint | 0/4 | 3/4 | FAIL |
| die | 3/4 | 4/4 | FAIL |
| multi_image | 2/4 | 3/4 | FAIL |

## Family results

| Family | Eligible? | Frozen confirmed? | Headline confirmed? | Control/component | Probe | Gap | Note |
|---|---|---|---|---:|---:|---:|---|
| B1_zigzag_direction | YES | NO | **NO** | 10/10 | 6/10 | 40.0 pp | eligible |
| B2_hidden_reconstruction | YES | YES | **YES** | 10/10 | 0/10 | 100.0 pp | eligible |
| B3_endpoint_anchoring | NO | NO | **NO** | 7/10 | 0/10 | 70.0 pp | failed calibration: CAL_endpoint |
| B4_perception_to_action | NO | NO | **NO** | 1/10 | 0/1 | 10.0 pp | failed calibration: CAL_endpoint |
| B5_multi_image_position | NO | NO | **NO** | 2/4 | 8/16 | 